In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_excel('Tabela_3_Areas_gerais_de_formacao_na_graduacao.xlsx', 
                   header=1,  # Pula a linha 0 que é o título
                   skipfooter=1)  # Pula a última linha (Fonte)


print("=== Estrutura inicial dos dados ===\n")
print(df.head())
print("\nColunas:", df.columns.tolist())


df.columns = ['Regiao', 'Total', 'Homens_Total', 'Mulheres_Total', 
              'CTEM_Total', 'CTEM_Homens', 'CTEM_Mulheres',
              'Educ_Saude_Total', 'Educ_Saude_Homens', 'Educ_Saude_Mulheres']




regioes = ['Brasil', 'Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']
estados = df[~df['Regiao'].isin(regioes)]['Regiao'].tolist()

print(f"\nRegiões: {regioes}")
print(f"Estados: {estados[:5]}... (total: {len(estados)} estados)")


cols_numericas = ['Total', 'Homens_Total', 'Mulheres_Total', 
                  'CTEM_Total', 'CTEM_Homens', 'CTEM_Mulheres',
                  'Educ_Saude_Total', 'Educ_Saude_Homens', 'Educ_Saude_Mulheres']

for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df_total_regioes = df[df['Regiao'].isin(regioes)].copy()
df_total_regioes = df_total_regioes[['Regiao', 'Homens_Total', 'Mulheres_Total']]

# 2.2 Homens e mulheres por região com curso nas áreas CTEM
df_ctem_regioes = df[df['Regiao'].isin(regioes)].copy()
df_ctem_regioes = df_ctem_regioes[['Regiao', 'CTEM_Homens', 'CTEM_Mulheres']]

# 2.3 Homens e mulheres por região com curso nas áreas Educação, Serviços e Saúde
df_educ_saude_regioes = df[df['Regiao'].isin(regioes)].copy()
df_educ_saude_regioes = df_educ_saude_regioes[['Regiao', 'Educ_Saude_Homens', 'Educ_Saude_Mulheres']]

print("\n=== Dados para análise por região ===\n")
print("Total de formados por região:\n", df_total_regioes)
print("\nCTEM por região:\n", df_ctem_regioes)
print("\nEducação, Serviços e Saúde por região:\n", df_educ_saude_regioes)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))


df_total_regioes_melted = df_total_regioes.melt(id_vars=['Regiao'], 
                                                 var_name='Sexo', 
                                                 value_name='Quantidade')
df_total_regioes_melted['Sexo'] = df_total_regioes_melted['Sexo'].map({
    'Homens_Total': 'Homens', 
    'Mulheres_Total': 'Mulheres'
})

sns.barplot(data=df_total_regioes_melted, x='Regiao', y='Quantidade', hue='Sexo', ax=axes[0,0])
axes[0,0].set_title('Total de pessoas com curso superior por região e sexo')
axes[0,0].set_xlabel('Região')
axes[0,0].set_ylabel('Quantidade')
axes[0,0].tick_params(axis='x', rotation=45)


df_ctem_melted = df_ctem_regioes.melt(id_vars=['Regiao'], 
                                      var_name='Sexo', 
                                      value_name='Quantidade')
df_ctem_melted['Sexo'] = df_ctem_melted['Sexo'].map({
    'CTEM_Homens': 'Homens', 
    'CTEM_Mulheres': 'Mulheres'
})

sns.barplot(data=df_ctem_melted, x='Regiao', y='Quantidade', hue='Sexo', ax=axes[0,1])
axes[0,1].set_title('CTEM - Ciência, Tecnologia, Engenharias e Matemática')
axes[0,1].set_xlabel('Região')
axes[0,1].set_ylabel('Quantidade')
axes[0,1].tick_params(axis='x', rotation=45)

df_educ_melted = df_educ_saude_regioes.melt(id_vars=['Regiao'], 
                                            var_name='Sexo', 
                                            value_name='Quantidade')
df_educ_melted['Sexo'] = df_educ_melted['Sexo'].map({
    'Educ_Saude_Homens': 'Homens', 
    'Educ_Saude_Mulheres': 'Mulheres'
})

sns.barplot(data=df_educ_melted, x='Regiao', y='Quantidade', hue='Sexo', ax=axes[1,0])
axes[1,0].set_title('Educação, Serviços pessoais, Saúde e Bem-estar')
axes[1,0].set_xlabel('Região')
axes[1,0].set_ylabel('Quantidade')
axes[1,0].tick_params(axis='x', rotation=45)


df_proporcao = df[df['Regiao'].isin(regioes)].copy()
df_proporcao['Pct_CTEM'] = (df_proporcao['CTEM_Total'] / df_proporcao['Total']) * 100
df_proporcao['Pct_Educ_Saude'] = (df_proporcao['Educ_Saude_Total'] / df_proporcao['Total']) * 100
df_proporcao['Pct_Outras'] = 100 - df_proporcao['Pct_CTEM'] - df_proporcao['Pct_Educ_Saude']

df_proporcao_melted = df_proporcao.melt(id_vars=['Regiao'], 
                                        value_vars=['Pct_CTEM', 'Pct_Educ_Saude', 'Pct_Outras'],
                                        var_name='Area', 
                                        value_name='Porcentagem')

df_proporcao_melted['Area'] = df_proporcao_melted['Area'].map({
    'Pct_CTEM': 'CTEM',
    'Pct_Educ_Saude': 'Educação/Saúde',
    'Pct_Outras': 'Outras áreas'
})

sns.barplot(data=df_proporcao_melted, x='Regiao', y='Porcentagem', hue='Area', ax=axes[1,1])
axes[1,1].set_title('Proporção de formados por área de conhecimento (%)')
axes[1,1].set_xlabel('Região')
axes[1,1].set_ylabel('Porcentagem (%)')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('analise_regioes_formacao.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("\n" + "="*60)
print("ESTATÍSTICAS DESCRITIVAS")
print("="*60)

# Total por região
print("\n=== Total de formados por região ===")
print(df_total_regioes.set_index('Regiao'))

# CTEM por região
print("\n=== CTEM - Ciência, Tecnologia, Engenharias e Matemática ===")
print(df_ctem_regioes.set_index('Regiao'))

# Educação/Saúde por região
print("\n=== Educação, Serviços pessoais, Saúde e Bem-estar ===")
print(df_educ_saude_regioes.set_index('Regiao'))

In [ ]:
print("\n" + "="*60)
print("PROPORÇÃO DE HOMENS E MULHERES POR ÁREA")
print("="*60)

for regiao in regioes:
    dados_regiao = df[df['Regiao'] == regiao].iloc[0]
    print(f"\n--- {regiao} ---")
    print(f"Total: Homens={dados_regiao['Homens_Total']:,} | Mulheres={dados_regiao['Mulheres_Total']:,}")
    print(f"  % Mulheres no total: {dados_regiao['Mulheres_Total']/dados_regiao['Total']*100:.1f}%")
    print(f"CTEM: Homens={dados_regiao['CTEM_Homens']:,} | Mulheres={dados_regiao['CTEM_Mulheres']:,}")
    print(f"  % Mulheres no CTEM: {dados_regiao['CTEM_Mulheres']/dados_regiao['CTEM_Total']*100:.1f}%")
    print(f"Educ/Saúde: Homens={dados_regiao['Educ_Saude_Homens']:,} | Mulheres={dados_regiao['Educ_Saude_Mulheres']:,}")
    print(f"  % Mulheres em Educ/Saúde: {dados_regiao['Educ_Saude_Mulheres']/dados_regiao['Educ_Saude_Total']*100:.1f}%")

In [ ]:
print("\n" + "="*60)
print("DESEQUILÍBRIO DE GÊNERO POR ÁREA (Mulheres - Homens)")
print("="*60)

df_desequilíbrio = df[df['Regiao'].isin(regioes)].copy()
df_desequilíbrio['Diferenca_Total'] = df_desequilíbrio['Mulheres_Total'] - df_desequilíbrio['Homens_Total']
df_desequilíbrio['Diferenca_CTEM'] = df_desequilíbrio['CTEM_Mulheres'] - df_desequilíbrio['CTEM_Homens']
df_desequilíbrio['Diferenca_Educ'] = df_desequilíbrio['Educ_Saude_Mulheres'] - df_desequilíbrio['Educ_Saude_Homens']

print(df_desequilíbrio[['Regiao', 'Diferenca_Total', 'Diferenca_CTEM', 'Diferenca_Educ']].set_index('Regiao'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Preparar dados para boxplots
dados_boxplot = df[df['Regiao'].isin(regioes)].copy()

# Boxplot Total
axes[0].boxplot([dados_boxplot['Homens_Total'], dados_boxplot['Mulheres_Total']], 
                labels=['Homens', 'Mulheres'])
axes[0].set_title('Distribuição de formados por região - Total')
axes[0].set_ylabel('Quantidade')

# Boxplot CTEM
axes[1].boxplot([dados_boxplot['CTEM_Homens'], dados_boxplot['CTEM_Mulheres']], 
                labels=['Homens', 'Mulheres'])
axes[1].set_title('Distribuição - CTEM')
axes[1].set_ylabel('Quantidade')

# Boxplot Educação/Saúde
axes[2].boxplot([dados_boxplot['Educ_Saude_Homens'], dados_boxplot['Educ_Saude_Mulheres']], 
                labels=['Homens', 'Mulheres'])
axes[2].set_title('Distribuição - Educação, Serviços e Saúde')
axes[2].set_ylabel('Quantidade')

plt.tight_layout()
plt.savefig('boxplot_distribuicao_regioes.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("ANÁLISE CONCLUÍDA")
print("="*60)